RAG (Retrival Augmented Generation)

# 📄 Build Your Own RAG Chatbot with LangChain

In this notebook you will build a chatbot that can answer questions
about any PDF you upload.

This is called RAG — Retrieval Augmented Generation.
Instead of the AI guessing from its training data, it reads YOUR
document and answers based on what is actually written in it.

Here is what will happen step by step:
1. You upload a PDF
2. We split it into small chunks
3. We convert those chunks into numbers (embeddings)
4. We store those numbers in a searchable database
5. When you ask a question, we find the most relevant chunks
6. We send those chunks + your question to GPT-4o
7. GPT-4o answers using only what is in your PDF

Let's build it.


# Install Libraries

In [1]:
!pip install langchain langchain-community langchain-openai faiss-cpu pypdf openai tiktoken langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency

# What We Just Installed

Here is what each library does:

langchain → the main framework that connects everything together

langchain-openai → lets LangChain talk to OpenAI's GPT-4o

langchain-community → extra tools for loading PDFs and other files

faiss-cpu → a fast database that stores and searches embeddings

pypdf → reads and extracts text from PDF files

tiktoken → counts tokens so we don't exceed GPT-4o's limit

# Set Your OpenAI API Key

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Upload Your PDF

In [3]:
from google.colab import files

print("Click the button below to upload your PDF")
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {pdf_filename}")

Click the button below to upload your PDF


Saving mitres_18_001_f17_full_book.pdf to mitres_18_001_f17_full_book.pdf
✅ Uploaded: mitres_18_001_f17_full_book.pdf


# Load and Split the PDF


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step 1 — Load the PDF
loader = PyPDFLoader(pdf_filename)
pages = loader.load()
print(f"📄 Total pages loaded: {len(pages)}")

# Step 2 — Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_documents(pages)
print(f"✂️ Total chunks created: {len(chunks)}")
print(f"\nExample chunk:\n{chunks[0].page_content[:300]}")

📄 Total pages loaded: 705
✂️ Total chunks created: 2393

Example chunk:
CALCULUS
Gilbert Strang
Massachuset
ts Institute of Technology
WELLES
LEY-CAMBRIDGE PRESS
Box 812060 Wellesley MA 02482


# Create Embeddings and Store in FAISS

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

print("⏳ Creating embeddings... please wait")

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
vector_store = FAISS.from_documents(chunks, embeddings)

print("✅ Embeddings created and stored successfully")
print(f"📦 Total chunks stored: {vector_store.index.ntotal}")

⏳ Creating embeddings... please wait
✅ Embeddings created and stored successfully
📦 Total chunks stored: 2393


# Build the Retriever and Chain

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

prompt = PromptTemplate(
    template="""
You are a helpful assistant. You remember the full conversation
history and also have access to a document.

Use conversation history for personal questions like the user's
name or anything they told you before.
Use the document context for questions about the document.
If neither has the answer, say so honestly.
Never mention that you have no conversation history or that
this is the first message. Just respond naturally.

Conversation History:
{chat_history}

Document Context:
{context}

Current Question:
{question}

Answer:
""",
    input_variables=["chat_history", "context", "question"]
)

chain = prompt | llm | StrOutputParser()

chat_history_store = []

def ask(question):
    history_text = ""
    for human, ai in chat_history_store:
        history_text += f"Human: {human}\nAssistant: {ai}\n"

    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    answer = chain.invoke({
        "chat_history": history_text,
        "context": context,
        "question": question
    })

    chat_history_store.append((question, answer))
    return answer, docs

print("✅ Chatbot with memory is ready")

✅ Chatbot with memory is ready


# Ask a Question

In [7]:
question = "What is the main topic of this document?"

answer, docs = ask(question)

print("🤖 Answer:")
print(answer)

print("\n📄 Sources used:")
for i, doc in enumerate(docs):
    print(f"\nSource {i+1} — Page {doc.metadata.get('page', 'unknown')}:")
    print(doc.page_content[:200])

🤖 Answer:
The main topic of the document is an introduction to a calculus book that emphasizes the importance of understanding mathematical concepts and ideas rather than just focusing on equations. It discusses the dynamic nature of mathematics, the relevance of patterns and functions, and the author's hope that students will find the explanations clear and engaging. The document also touches on the author's intention to inspire students and invites feedback on the book.

📄 Sources used:

Source 1 — Page 9:
To the Student
I ho
pe you will learn calculus from this book. On this page I will a d mit that I even
hope for more. If you ﬁnd that the explanations are clear, and also the purpose is
clear—that mea

Source 2 — Page 9:
comes down to teaching a person.
It is not easy to stay inspired for a year—probably impossible. But mathematics
is more active and cheerful than most people know. This book was written in a happy
spi

Source 3 — Page 167:
what is important. The main point of the o

# Build the Gradio Chatbot Interface

In [ ]:
import gradio as gr

def chat(question, history):
    if not question.strip():
        return history, history, ""

    answer, docs = ask(question)

    sources = []
    for doc in docs:
        page = doc.metadata.get("page", "unknown")
        snippet = doc.page_content[:150].replace("\n", " ")
        sources.append(f"📄 Page {page}: {snippet}...")

    sources_text = "\n\n".join(sources)
    full_answer = f"{answer}\n\n---\n**Sources used:**\n{sources_text}"

    history.append((question, full_answer))

    return history, history, ""

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 💬 RAG Chatbot with Memory")
    gr.Markdown("Ask anything about your PDF.")

    chatbot = gr.Chatbot(
        label="Chat",
        height=500,
        bubble_full_width=False
    )

    with gr.Row():
        question_box = gr.Textbox(
            placeholder="Type your question here and press Enter...",
            label="Your Question",
            scale=8,
            autofocus=True
        )
        submit_btn = gr.Button(
            "Ask",
            variant="primary",
            scale=1
        )

    gr.ClearButton(
        components=[chatbot, question_box],
        value="🗑️ Clear Chat"
    )

    state = gr.State([])

    question_box.submit(
        fn=chat,
        inputs=[question_box, state],
        outputs=[chatbot, state, question_box]
    )

    submit_btn.click(
        fn=chat,
        inputs=[question_box, state],
        outputs=[chatbot, state, question_box]
    )

demo.launch(debug=True)

/tmp/ipykernel_4208/2137147934.py:22: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_4208/2137147934.py:27: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_4208/2137147934.py:27: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_4208/2137147934.py:27: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=F

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://54e7daf5bdb4c1e071.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
